In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from sklearn.metrics import f1_score

In [8]:
def build_model(num_classes=7, backbone='resnet50', pretrained=True):
    if backbone == 'resnet50':
        model = models.resnet50(weights='IMAGENET1K_V2' if pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise NotImplementedError(f"Unsupported backbone: {backbone}")
    return model

In [3]:
class Explainable_model(nn.Module):
    def __init__(self, network, num_classes, pretrained=True):
        super(Explainable_model, self).__init__()

        if network == 'resnet50':
            model = models.resnet50(weights='IMAGENET1K_V2' if pretrained else None)
            backbone = list(model.children())[:-2]
            classifier = nn.Conv2d(model.fc.in_features, num_classes, kernel_size=(1,1), stride=1)
        else:
            raise ValueError(f"Unsupported backbone: {network}. To be implemented")

        self.backbone = nn.Sequential(*backbone)
        self.classifier = classifier

    def forward(self, x):        
        x = self.backbone(x)                # (bs, c, h, w) 
        activation = self.classifier(x)     # (bs, n_class, h, w) 
        bs, c, h, w = x.shape               # (bs, n_class, h, w) 
          
        avgpool = nn.AvgPool2d(kernel_size=(h, w), stride=(1,1), padding=0)             
        out = avgpool(activation)           # (bs, n_class, 1, 1) 
        out = out.view(out.shape[0], -1)    # (bs, n_class)
        
        return out, activation

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = build_model(num_classes=7, backbone='resnet50').to(device)

In [5]:
xai_model = Explainable_model(network='resnet50', num_classes=7)

## Train and val loader

In [ ]:
[ToDo]

## Training loop
With per-epoch train/val tracking and checkpointing on best validation macro-F1 (a better metric than accuracy here, since accuracy is misleading under this level of class imbalance):

In [6]:
criterion = nn.CrossEntropyLoss()  # no weight= needed since we're using the sampler
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

In [ ]:
criterion = nn.CrossEntropyLoss()  # no weight= needed since you're using the sampler
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, macro_f1, all_preds, all_labels

# training loop
num_epochs = 20
best_f1 = 0.0

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  -> Saved new best model (F1: {best_f1:.4f})")

## ToDo by Team
- In `build_model` and `Explainable_model`, add **EfficientNet_B0** and **BagNet-33**